<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/04_download_satellite_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GridSat-B1 infrared images for every case

For every case (storm observation) in the development and test case files, this notebook:

1. takes the storm's position and time at that observation,
2. matches the time to the GridSat-B1 file at or just before it (GridSat has one global file every 3 hours, so the image never comes from after t),
3. downloads each global file once (about 30–40 MB), even when several cases share it, and deletes it after use,
4. cuts three storm-centred crops (wide, mid, core) around the grid cell nearest the storm centre, each square in kilometres (about 2,340, 1,170 and 580 km across),
5. writes them as plain infrared PNG images, with nothing drawn on them, into `model_inputs/images/wide/`, `model_inputs/images/mid/` and `model_inputs/images/core/`,
6. records every case in a manifest (`model_inputs/images/manifest.csv`): its status, the GridSat time used and how much data each crop is missing.

Images are named `<case_id>_<crop>.png`, for example `images/mid/2016240N13224_007_mid.png`, relative to the `model_inputs/` folder. The case files already hold exactly these paths (`image_t_wide`, `image_t_mid`, `image_t_core`), so this notebook never changes the case files. Whether an image was actually made is recorded only in the manifest.

It can be stopped and re-run: cases already done are skipped.

**Size of the job:** the 2,485 cases need about 2,100 GridSat files, roughly 73 GB of downloads in total (each file is deleted after use, so disk use stays small). Try a few cases first with `LIMIT`.

## 1. Setup

The next cell works out the repo folder and makes `src/` importable, as in the other notebooks. In Colab it first clones this repo. It also installs `netCDF4`, which reads the GridSat files, if it is missing.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

try:
    import netCDF4  # noqa: F401  (reads the GridSat files)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "netCDF4"], check=True)

## 2. Inputs

**Cases.** Both RI case files in `model_inputs/ri_prediction/` (the identification probe and the image prompts use the same cases, so they need no images of their own). Each case's position (`lat`, `lon`) comes from the data CSV, matched on storm ID and time.

**Crop sizes** are the north–south size in GridSat pixels (0.07 degrees of latitude, about 7.8 km each) and must be odd so the storm centre is the middle pixel. GridSat pixels are 0.07 degrees of longitude wide, which is fewer kilometres away from the equator, so each crop takes `size / cos(latitude)` pixels east–west (rounded to an odd number). After resizing to `PNG_SIZE` × `PNG_SIZE`, every image therefore covers the same distance north–south and east–west, at any latitude, and its shape gives no hint of latitude.

**Missing data** (pixels where the satellite has no data) is drawn in `MISSING_RGB`, magenta by default, a colour that can't be mistaken for any temperature on the grey scale. The manifest records how much of each crop is missing.

**Brightness scale** is fixed (`VMIN` to `VMAX` kelvin), so the same grey always means the same temperature in every image. Cold cloud tops are white.

**Where images go.** By default into `model_inputs/images/` in the repo. In Colab, set `USE_DRIVE = True` to save them to Google Drive instead, so they survive a disconnect. Afterwards, copy the Drive folder's `images/` into the repo's `model_inputs/` folder wherever you want to use the images.

In [ ]:
import pandas as pd

# ---- cases ----------------------------------------------------------------------
CASE_FILES = [
    ROOT / "model_inputs" / "ri_prediction" / "dev" / "dev_2016_20ri_20nori_text_only.jsonl.gz",
    ROOT / "model_inputs" / "ri_prediction" / "test" / "test_2017-2025_50ri_50nori_text_only.jsonl.gz",
]
DATA_CSV = ROOT / "data" / "ibtracs_era5_sat_2016_2025.csv"
LIMIT = None               # e.g. 20 to try the first 20 cases only

# ---- images -----------------------------------------------------------------------
CROP_SIZES_PX = {"wide": 301, "mid": 151, "core": 75}   # north-south pixels: about 2,340, 1,170 and 580 km
PNG_SIZE = 512             # every PNG is resized to PNG_SIZE x PNG_SIZE
VARIABLE = "irwin_cdr"     # 11 micron infrared window brightness temperature (K)
VMIN, VMAX = 190.0, 310.0  # fixed brightness-temperature range (K)
COLORMAP = "gray_r"        # cold = white
MISSING_RGB = (255, 0, 255)     # magenta where the satellite has no data (not a temperature on the grey scale)

# ---- output and download ----------------------------------------------------------
USE_DRIVE = False          # True saves to Google Drive (Colab), so results survive a disconnect
DRIVE_DIR = "wacv2027_output"   # folder inside MyDrive when USE_DRIVE is True
MAX_TIME_OFFSET_MIN = 180  # the GridSat time used is at most this many minutes before the case time
DOWNLOAD_WORKERS = 8       # files downloaded in parallel
BATCH_FILES = 16           # files kept on disk at once (about 30-40 MB each)

## 3. Output folders

`out_dir` holds `images/wide/`, `images/mid/`, `images/core/` and the manifest (`images/manifest.csv`). Downloaded GridSat files are kept briefly in `.gridsat_tmp/` and deleted after use.

In [ ]:
from prompts import IMAGE_CROPS, image_relpath

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = Path("/content/drive/MyDrive") / DRIVE_DIR
else:
    out_dir = ROOT / "model_inputs"

tmp_dir = ROOT / ".gridsat_tmp"
manifest_path = out_dir / "images" / "manifest.csv"

assert sorted(CROP_SIZES_PX) == sorted(IMAGE_CROPS), "crop names must match IMAGE_CROPS in src/prompts.py"
assert all(s % 2 == 1 for s in CROP_SIZES_PX.values()), "crop sizes must be odd numbers"
for crop in CROP_SIZES_PX:
    (out_dir / "images" / crop).mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(exist_ok=True)
print("Saving images to", out_dir / "images")

## 4. Match every case to a GridSat time

Each case's time is matched to the GridSat file **at or before** it (rounded down to the 3-hour grid), so no image comes from after the case time. Longitude is converted to the grid's -180 to 180 convention. Check the summary before starting the download.

In [ ]:
import numpy as np

missing = [str(f.relative_to(ROOT)) for f in CASE_FILES if not f.exists()]
assert not missing, f"case files not found: {missing}. Run 02_make_prompt_files.ipynb, or update this clone with git pull."

# convert_dates=False keeps iso_time as text, exactly as in the CSV (pandas would otherwise turn *_time columns into dates).
cases = pd.concat([pd.read_json(f, lines=True, convert_dates=False, dtype={"sid": str, "iso_time": str, "case_id": str})
                   for f in CASE_FILES], ignore_index=True)
assert cases["case_id"].is_unique, "a case_id appears twice across the case files"

positions = pd.read_csv(DATA_CSV, usecols=["sid", "iso_time", "lat", "lon"], dtype={"sid": str, "iso_time": str})
cases = cases[["case_id", "split", "sid", "iso_time"]].merge(positions, on=["sid", "iso_time"], how="left", validate="one_to_one")
assert cases[["lat", "lon"]].notna().all().all(), "some cases have no position in the data CSV"

points = pd.DataFrame({
    "id": cases["case_id"],
    "sid": cases["sid"],
    "split": cases["split"],
    "storm_time": pd.to_datetime(cases["iso_time"]),
    "storm_lat": cases["lat"],
    "storm_lon": ((cases["lon"] + 180) % 360) - 180,
})
points["gridsat_time"] = points["storm_time"].dt.floor("3h")          # at or before the case time
points["offset_min"] = (points["storm_time"] - points["gridsat_time"]).dt.total_seconds() / 60

if LIMIT:
    points = points.head(LIMIT)

n_files = points["gridsat_time"].nunique()
print(f"Cases                  : {len(points)}  ({points['split'].value_counts().to_dict()})")
print(f"Period                 : {points['storm_time'].min()} to {points['storm_time'].max()}")
print(f"Cases off the 3 h grid : {int((points['offset_min'] > 0).sum())} (largest offset {points['offset_min'].max():.0f} min)")
print(f"GridSat files needed   : {n_files}  (about {n_files * 0.035:.0f} GB to download, deleted as it goes)")
points.head()

## 5. Functions

Download one file, read it, cut the crops, write the PNGs. Each crop is `size` pixels north–south and `size / cos(latitude)` pixels east–west, so it is square in kilometres. Crops that run past the grid's north or south edge (70 degrees) are padded with missing data rather than dropped, and columns wrap around the dateline.

In [ ]:
import shutil
import time
import urllib.error
import urllib.request

import matplotlib
import xarray as xr
from PIL import Image


def gridsat_file_name(t):
    return f"GRIDSAT-B1.{t:%Y.%m.%d.%H}.v02r01.nc"


def download_file(t):
    """Download the global file for GridSat time t. Returns (path, "ok") or (None, reason)."""
    name = gridsat_file_name(t)
    path = tmp_dir / name
    if path.exists():
        return path, "ok"
    urls = [
        f"https://noaa-cdr-gridsat-b1-pds.s3.amazonaws.com/data/{t:%Y}/{name}",
        "https://www.ncei.noaa.gov/data/geostationary-ir-channel-brightness-temperature-gridsat-b1/"
        f"access/{t:%Y}/{name}",
    ]
    part = path.with_name(name + ".part")
    reason = "no_file"                       # stays "no_file" only if every server says it does not exist
    for url in urls:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(url, timeout=120) as response, open(part, "wb") as out:
                    shutil.copyfileobj(response, out)
                os.replace(part, path)
                return path, "ok"
            except urllib.error.HTTPError as err:
                if err.code in (403, 404):
                    break                    # not on this server, try the next one
                reason = "download_failed"
            except Exception:
                reason = "download_failed"
            time.sleep(2 * (attempt + 1))
    return None, reason


def load_field(path):
    """Read one GridSat file. Returns 1-D lat, 1-D lon and the 2-D field (lat, lon) in kelvin."""
    with xr.open_dataset(path) as ds:
        if VARIABLE not in ds:
            raise KeyError(f"{VARIABLE!r} is not in the file. Available: {list(ds.data_vars)}")
        field = ds[VARIABLE].isel(time=0).transpose("lat", "lon").values
        return ds["lat"].values, ds["lon"].values, field


def width_px(size, storm_lat):
    """East-west pixels that span the same distance as `size` pixels north-south at this latitude (odd)."""
    width = int(round(size / np.cos(np.radians(storm_lat))))
    return width if width % 2 == 1 else width + 1


def crop_storm(field, lat, lon, storm_lat, storm_lon, n_rows, n_cols):
    """Cut an n_rows x n_cols window centred on the storm, north at the top, NaN where there is no data."""
    iy = int(np.abs(lat - storm_lat).argmin())
    ix = int(np.abs(((lon - storm_lon + 180) % 360) - 180).argmin())   # nearest across the dateline too

    rows = np.arange(iy - n_rows // 2, iy + n_rows // 2 + 1)
    cols = np.arange(ix - n_cols // 2, ix + n_cols // 2 + 1) % lon.size   # wrap around the dateline
    inside = (rows >= 0) & (rows < lat.size)                            # rows past 70 N/S stay NaN

    image = np.full((n_rows, n_cols), np.nan, dtype="float32")
    image[inside] = field[np.ix_(rows[inside], cols)]
    if lat[1] > lat[0]:                                                 # file runs south to north
        image = image[::-1]
    return image, iy, ix


def center_crop(image, n_rows, n_cols):
    """The central n_rows x n_cols part of image (both odd, so the storm stays in the middle pixel)."""
    top = (image.shape[0] - n_rows) // 2
    left = (image.shape[1] - n_cols) // 2
    return image[top:top + n_rows, left:left + n_cols]


def to_png(image, path):
    """Write a plain image: fixed temperature scale, nothing drawn on it."""
    scaled = np.clip((image - VMIN) / (VMAX - VMIN), 0, 1)
    rgb = (matplotlib.colormaps[COLORMAP](np.nan_to_num(scaled))[..., :3] * 255).astype("uint8")
    rgb[np.isnan(image)] = MISSING_RGB
    Image.fromarray(rgb).resize((PNG_SIZE, PNG_SIZE), Image.BICUBIC).save(path)


def process_point(row, lat, lon, field):
    """Make all crops for one case. Returns the manifest fields for it."""
    if not lat.min() <= row.storm_lat <= lat.max():
        return {"status": "outside_grid"}

    widest = max(CROP_SIZES_PX.values())
    big, iy, ix = crop_storm(field, lat, lon, row.storm_lat, row.storm_lon, widest, width_px(widest, row.storm_lat))

    record = {"status": "ok", "grid_lat": round(float(lat[iy]), 3), "grid_lon": round(float(lon[ix]), 3)}
    for name, size in CROP_SIZES_PX.items():
        image = center_crop(big, size, width_px(size, row.storm_lat))   # square in km
        rel = image_relpath(row.id, name)                    # images/<crop>/<case_id>_<crop>.png
        to_png(image, out_dir / rel)
        record[f"png_{name}"] = rel
        record[f"missing_pct_{name}"] = round(100 * float(np.isnan(image).mean()), 1)
    record["min_temp_k"] = round(float(np.nanmin(big)), 1) if np.isfinite(big).any() else np.nan
    return record

## 6. Run

Files are downloaded a batch at a time, processed, and deleted. The manifest is written after every batch, so if the run stops you can simply run the notebook again and it continues where it stopped.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

BASE_COLS = ["id", "sid", "split", "storm_time", "gridsat_time", "offset_min", "storm_lat", "storm_lon"]
ID_TYPES = {"id": str, "sid": str}


def base(row):
    return {c: getattr(row, c) for c in BASE_COLS}


def save_records(records):
    """Append to the manifest, keeping the latest row for each case."""
    new = pd.DataFrame(records)
    if manifest_path.exists():
        new = pd.concat([pd.read_csv(manifest_path, dtype=ID_TYPES), new], ignore_index=True)
    new["id"] = new["id"].astype(str)
    new.drop_duplicates("id", keep="last").sort_values("id").to_csv(manifest_path, index=False)


done = set()
if manifest_path.exists():
    old = pd.read_csv(manifest_path, dtype=ID_TYPES)
    done = set(old.loc[old["status"].isin(["ok", "outside_grid"]), "id"])   # nothing more to do for these

todo = points[~points["id"].isin(done)]
too_far = todo["offset_min"] > MAX_TIME_OFFSET_MIN

records = [{**base(r), "status": "time_offset_too_large"} for r in todo[too_far].itertuples()]
if records:
    save_records(records)

groups = list(todo[~too_far].groupby("gridsat_time"))
print(f"{len(done)} cases already finished, {len(todo)} to go, {len(groups)} files to download")

with tqdm(total=len(groups), unit="file") as bar:
    for start in range(0, len(groups), BATCH_FILES):
        batch = groups[start:start + BATCH_FILES]
        with ThreadPoolExecutor(DOWNLOAD_WORKERS) as pool:
            downloads = list(pool.map(download_file, [t for t, _ in batch]))

        records = []
        for (t, group), (path, reason) in zip(batch, downloads):
            if path is None:
                records += [{**base(r), "status": reason} for r in group.itertuples()]
            else:
                try:
                    lat, lon, field = load_field(path)
                    records += [{**base(r), **process_point(r, lat, lon, field)} for r in group.itertuples()]
                except Exception as err:
                    records += [{**base(r), "status": f"error: {err}"} for r in group.itertuples()]
                os.remove(path)
            bar.update(1)
        save_records(records)

manifest = pd.read_csv(manifest_path, dtype=ID_TYPES)
print(manifest["status"].value_counts().to_string())

## 7. Check the result

The manifest has one row per case: the matched GridSat time, the grid cell used, the PNG paths, and how much of each crop is missing. The three images for the first finished case are shown below, exactly as saved.

In [ ]:
import matplotlib.pyplot as plt

ok = manifest[manifest["status"] == "ok"]
display(manifest.head())

if len(ok):
    first = ok.iloc[0]
    fig, axes = plt.subplots(1, len(CROP_SIZES_PX), figsize=(4.5 * len(CROP_SIZES_PX), 4.9), layout="constrained")
    for ax, (name, size) in zip(np.atleast_1d(axes), CROP_SIZES_PX.items()):
        ax.imshow(Image.open(out_dir / first[f"png_{name}"]))
        ax.set_title(f"{name}: about {size * 0.07 * 111.2:,.0f} km across")
        ax.axis("off")
    fig.suptitle(f"Case {first['id']}, GridSat {first['gridsat_time']} UTC")
    plt.show()

## Notes

- **File names and the model:** the case ID (storm ID + observation number) is in each file name. Send the model the image bytes only, never the file name or path, and the storm stays hidden.
- **Folders:** `model_inputs/images/wide/`, `mid/` and `core/` each hold one PNG per case; `model_inputs/images/manifest.csv` lists every case and its status.
- **Statuses in the manifest:** `ok`; `no_file` (GridSat has no file for that time); `download_failed` (network trouble, re-run to retry); `outside_grid` (centre poleward of 70 degrees); `time_offset_too_large`.
- **Missing data:** pixels without satellite data are drawn in `MISSING_RGB` (magenta), which can't be confused with a real temperature. Use the `missing_pct_*` columns to find or drop images with large gaps, and tell the model what magenta means if you keep such images.
- **Square in kilometres:** each crop spans the same distance east–west as north–south, measured at the storm centre's latitude, so image shape doesn't depend on latitude.
- **Using the images:** the case files point to every image, made or not. Before using an image, join the case file with the manifest on `case_id` (= the manifest's `id`) and keep cases with `status == "ok"`; see the README.